# ComplaintIQ - unsupervised model at full scale (`07b_mllib_unsupervised`)

`05` clustered a 25k **sample** in sklearn. This notebook clusters **every narrative** (~3.8M) with
distributed Spark MLlib `KMeans` - no sampling. Companion to **`07a_mllib_supervised.ipynb`**; split
so each model trains in its own session and stays under the serverless Spark-Connect **1GB
model-cache cap**.

- **Goal:** recover complaint themes. **Metric:** cluster purity vs the `product x issue` yardstick.

## How to read this notebook
Everything stays in Spark; only scalar metrics reach the driver. We load just the three columns the
clustering needs (narrative + the two yardstick fields), so the DataFrame shipped to Spark Connect
is small.

> **What the "product x issue" yardstick is:** each complaint has `product` (what it is about) and
> `issue` (the problem); their combination labels a real theme (`03` section 8). Clustering has no
> label, so we measure how concentrated each cluster is on one theme - a measuring stick, never a
> feature.

> **Go deeper:**
> - [Spark ML clustering](https://spark.apache.org/docs/latest/ml-clustering.html): *KMeans k / maxIter / initMode and ClusteringEvaluator (silhouette), ~10 min.*
> - [Spark ML feature extractors](https://spark.apache.org/docs/latest/ml-features.html): *Tokenizer, HashingTF, IDF - the text representation used here, ~15 min.*

## Setup

In [ ]:
import sys
from pathlib import Path

# Import shared helpers from the local complaintiq package. Walk up from the cwd
# to find the repo's src/ dir, so this works whether the notebook lives in
# notebooks/ or notebooks/appendix/, locally or in a Databricks Git folder.
_here = Path.cwd()
_src = None
for _p in [_here, *_here.parents]:
    if (_p / "src" / "complaintiq").exists():
        _src = str(_p / "src")
        break
if _src and _src not in sys.path:
    sys.path.insert(0, _src)

In [ ]:
from complaintiq import RANDOM_STATE, np, pd, plt, sns, print_versions  # shared setup
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, HashingTF, IDF
from pyspark.ml.clustering import KMeans

RANDOM_STATE = 42
print("Spark", spark.version)

---
## 1. Load only narrative + the yardstick fields

Prefer the narrative-only Parquet (dense text). We need `complaint_text`, `product`, and `issue` -
nothing else - so the projection stays narrow.

In [ ]:
from pathlib import Path

VOLUME_DIR = Path("/Volumes/workspace/complaintiq/data")
data_dir = VOLUME_DIR
if not data_dir.exists():
    # Local: walk up from cwd to find the repo's data/ dir (robust to notebook depth).
    for _p in [Path.cwd(), *Path.cwd().parents]:
        if (_p / "data").is_dir():
            data_dir = _p / "data"
            break
nar_path = data_dir / "complaints_narrative_only.parquet"
full_path = data_dir / "complaints.parquet"
src = str(nar_path) if nar_path.exists() else str(full_path)

docs = spark.read.parquet(src)
if "has_narrative" in docs.columns and src == str(full_path):
    docs = docs.filter(F.col("has_narrative"))
docs = (
    docs.select("complaint_text", "product", "issue")
    .dropna(subset=["complaint_text", "product"])
    .filter(F.length("complaint_text") > 0)
    .withColumn("theme", F.concat_ws(" | ", "product", "issue"))
)
print(f"narrative corpus: {docs.count():,} rows")

---
## 2. Represent + cluster the full corpus

Hashed TF-IDF (`Tokenizer` -> `HashingTF` -> `IDF`) then MLlib `KMeans` at k = number of products
(the same principled, tuning-free k as `05`). Runs over every narrative, distributed.

In [ ]:
pipeline = Pipeline(
    stages=[
        Tokenizer(inputCol="complaint_text", outputCol="tokens"),
        HashingTF(inputCol="tokens", outputCol="term_freq", numFeatures=2**16),
        IDF(inputCol="term_freq", outputCol="features"),
    ]
)
featurized = pipeline.fit(docs).transform(docs)

n_clusters = docs.select("product").distinct().count()
clustered = (
    KMeans(
        featuresCol="features", predictionCol="cluster", k=n_clusters, seed=RANDOM_STATE, maxIter=20
    )
    .fit(featurized)
    .transform(featurized)
)
print(f"clustered {docs.count():,} narratives into n_clusters={n_clusters} clusters")

---
## 3. Cluster purity vs the yardstick

Purity = the share of rows that sit in their cluster's **dominant** `product x issue` theme. All
group-by aggregations, nothing collected to the driver. It is a driver-free stand-in for the ARI/NMI
`05` reports (those need labels aligned in memory); it answers the same question - do the clusters
line up with real themes?

In [ ]:
counts = clustered.groupBy("cluster", "theme").count()
per = counts.groupBy("cluster").agg(F.max("count").alias("top"), F.sum("count").alias("total"))
purity = per.agg(F.sum("top")).first()[0] / per.agg(F.sum("total")).first()[0]
n_docs = docs.count()
print(f"KMeans k={n_clusters}")
print(
    f"cluster purity vs product x issue: {purity:.4f}  (share of rows in their cluster's dominant theme)"
)

# Persist metrics to the volume (notebook stdout is not exposed via the jobs API).
import json as _json, time

metrics = {
    "notebook": "07b_mllib_unsupervised",
    "rows": int(n_docs),
    "k": int(n_clusters),
    "cluster_purity": float(purity),
    "ts": time.strftime("%Y-%m-%dT%H:%M:%S"),
}
out = "/Volumes/workspace/complaintiq/data/metrics_07b.json"
dbutils.fs.put(out, _json.dumps(metrics, indent=2), overwrite=True)
print("wrote", out)

# A peek at the largest clusters' dominant themes (small result, safe to show).
from pyspark.sql.window import Window

rank_in_cluster = Window.partitionBy("cluster").orderBy(F.col("count").desc())
top_themes = (
    counts.withColumn("rk", F.row_number().over(rank_in_cluster))
    .filter(F.col("rk") == 1)
    .orderBy(F.col("count").desc())
    .limit(10)
)
display(top_themes.select("cluster", "theme", "count").toPandas())

> **What you're seeing:** full-corpus clustering purity against the label-free yardstick, plus the
> dominant theme of the largest clusters.
>
> **Why it matters:** `05`'s clustering at full scale. Purity answers the same question as ARI/NMI -
> do clusters recover real `product x issue` themes - without collecting labels to the driver.

---
## 4. Takeaways

> - **Scale:** clusters every narrative (~3.8M), not a 25k sample - no `.toPandas()` for the model.
> - **Metric:** cluster purity vs `product x issue` is the driver-free analog of `05`'s ARI/NMI.
> - **Small session:** one model, three loaded columns, hashed features - stays under the serverless
>   1GB ML-cache cap.
> - **Next:** the representation levers from `06` section 5 (dedup, LSA/UMAP, embeddings) apply here
>   too; `06` establishes they beat the raw-TF-IDF floor on the sample.

Full-data supervised model is in **`07a_mllib_supervised.ipynb`**.

> **Serverless limitation (measured 2026-08-03):** on Databricks Free Edition (Spark Connect serverless), fitting this full-corpus MLlib KMeans fails with `CONNECT_ML.MODEL_SIZE_OVERFLOW_EXCEPTION` - the fitted model transferred to the client is ~320MB, over the 256MB cap. The size is independent of `numFeatures` and sample size (500k and 3.8M both hit ~320MB); locally the saved model is ~1MB, so the overhead is a Spark Connect serialization artifact (the KMeans training summary), not the model parameters. Full-corpus MLlib clustering therefore runs on classic Spark but not on this serverless Connect path. The clustering story is carried by `05` (sklearn sample), `08` (embeddings sample), and `10` (embedding scaling); 07b is an appendix experiment. Run it on classic (non-serverless) compute to reproduce.